In [1]:
!pip install -q gradio opencv-python-headless pillow numpy matplotlib scipy mediapipe

In [2]:
!pip install mediapipe==1.2.2

In [3]:
!wget -q -O /content/blaze_face_short_range.tflite \
https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite

In [8]:
%%writefile /content/requirements.txt
gradio
opencv-python-headless
numpy
Pillow
scikit-image
mediapipe

Writing /content/requirements.txt


In [9]:
from google.colab import files

files.download('/content/requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

import cv2
import gradio as gr
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np


# ==========================================
# 共用：讀取影像
# ==========================================
def read_image(image_path):
    """讀取影像；讀取失敗時回傳 None。"""
    if image_path is None:
        return None
    return cv2.imread(str(image_path), cv2.IMREAD_COLOR)


# ==========================================
# 1. 影像載入與基礎解析 (Metadata)
# ==========================================
def get_image_info(image_path):
    if image_path is None:
        return "請先上傳影像", None

    img = read_image(image_path)
    if img is None:
        return "無法讀取影像，請確認格式 (PNG, JPG, BMP)", None

    height, width, channels = img.shape
    file_size = os.path.getsize(image_path) / 1024  # KB
    dtype = img.dtype

    info_text = (
        "### 📋 影像基本中繼資訊\n"
        f"- **解析度 (Resolution):** {width} × {height} 像素\n"
        f"- **色彩維度 (Channels):** {channels} 通道 (BGR)\n"
        f"- **資料型態 (Data Type):** {dtype}\n"
        f"- **檔案大小 (File Size):** {file_size:.2f} KB"
    )
    return info_text, img


# ==========================================
# 2. 多色彩空間轉換與同步直方圖分析
# ==========================================
def color_space_and_histogram(image_path, mode):
    if image_path is None:
        return None, None

    img_bgr = read_image(image_path)
    if img_bgr is None:
        return None, None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(6, 4))

    try:
        if mode == "Original (RGB)":
            display_img = img_rgb
            colors = ("r", "g", "b")
            for i, color in enumerate(colors):
                hist = cv2.calcHist([img_rgb], [i], None, [256], [0, 256])
                ax.plot(hist, color=color, label=f"{color.upper()} channel")
            ax.set_title("RGB Histogram")

        elif mode == "Red Channel (R)":
            display_img = np.zeros_like(img_rgb)
            display_img[:, :, 0] = img_rgb[:, :, 0]
            hist = cv2.calcHist([img_rgb], [0], None, [256], [0, 256])
            ax.plot(hist, color="r", label="R channel")
            ax.set_title("Red Channel Histogram")

        elif mode == "Green Channel (G)":
            display_img = np.zeros_like(img_rgb)
            display_img[:, :, 1] = img_rgb[:, :, 1]
            hist = cv2.calcHist([img_rgb], [1], None, [256], [0, 256])
            ax.plot(hist, color="g", label="G channel")
            ax.set_title("Green Channel Histogram")

        elif mode == "Blue Channel (B)":
            display_img = np.zeros_like(img_rgb)
            display_img[:, :, 2] = img_rgb[:, :, 2]
            hist = cv2.calcHist([img_rgb], [2], None, [256], [0, 256])
            ax.plot(hist, color="b", label="B channel")
            ax.set_title("Blue Channel Histogram")

        elif mode in (
            "YUV - Y (Luminance)",
            "YUV - U (Chrominance)",
            "YUV - V (Chrominance)",
        ):
            img_yuv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YUV)
            y, u, v = cv2.split(img_yuv)

            if mode == "YUV - Y (Luminance)":
                channel, color, title = y, "black", "Y (Luminance) Histogram"
            elif mode == "YUV - U (Chrominance)":
                channel, color, title = u, "blue", "U (Chrominance) Histogram"
            else:
                channel, color, title = v, "red", "V (Chrominance) Histogram"

            display_img = cv2.cvtColor(channel, cv2.COLOR_GRAY2RGB)
            hist = cv2.calcHist([channel], [0], None, [256], [0, 256])
            ax.plot(hist, color=color, label=title.split(" ")[0])
            ax.set_title(title)

        else:
            plt.close(fig)
            return None, None

        ax.set_xlim([0, 256])
        ax.set_xlabel("Pixel Intensity (0-255)")
        ax.set_ylabel("Pixel Count")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.6)

        # 新版 Matplotlib 使用 buffer_rgba()，再取前三個 RGB 通道。
        fig.canvas.draw()
        hist_img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()

    finally:
        plt.close(fig)

    return display_img, hist_img


# ==========================================
# 3. 自訂卷積核空間濾波與 PSNR 計算
# ==========================================

def apply_custom_filter(
    image_path, k1, k2, k3, k4, k5, k6, k7, k8, k9, normalize
):
    if image_path is None:
        return None, "請先上傳影像"

    img = read_image(image_path)
    if img is None:
        return None, "無法讀取影像，請確認檔案格式。"

    # 組合 3×3 卷積核
    kernel = np.array(
        [[k1, k2, k3], [k4, k5, k6], [k7, k8, k9]],
        dtype=np.float32,
    )

    # 權重正規化
    normalization_note = "未啟用"
    if normalize:
        kernel_sum = float(np.sum(kernel))
        if not np.isclose(kernel_sum, 0.0):
            kernel = kernel / kernel_sum
            normalization_note = "已啟用 (Normalized)"
        else:
            normalization_note = "已勾選，但卷積核總和為 0，因此未執行正規化"

    # 直接對彩色 BGR 影像進行卷積
    filtered_float = cv2.filter2D(img, cv2.CV_32F, kernel)

    # 限制像素值在 0～255
    filtered_bgr = np.clip(
        np.rint(filtered_float), 0, 255
    ).astype(np.uint8)

    # 轉換成 RGB，供 Gradio 顯示
    filtered_rgb = cv2.cvtColor(
        filtered_bgr, cv2.COLOR_BGR2RGB
    )

    # PSNR：分別比較原始彩色影像與濾波後影像
    mse = float(
        np.mean(
            (
                img.astype(np.float64)
                - filtered_bgr.astype(np.float64)
            ) ** 2
        )
    )

    if mse == 0:
        psnr_text = "∞ (兩張影像完全相同)"
    else:
        max_i = 255.0
        psnr_val = 10 * np.log10((max_i ** 2) / mse)
        psnr_text = f"{psnr_val:.2f} dB"

    kernel_text = np.array2string(
        kernel, precision=4, suppress_small=True
    )

    result_text = f"""### 📊 濾波與品質評估結果
- **使用的卷積核 (Kernel):**
```text
{kernel_text}
```
- **正規化狀態:** {normalization_note}
- **影像處理模式:** 彩色 RGB 影像 (三通道分別運算)
- **均方誤差 (MSE):** {mse:.4f}
- **峰值信噪比 (PSNR):** **{psnr_text}**

#### 💡 PSNR 計算原理
1. **均方誤差 (MSE)**：計算原圖 I 與濾波後影像 K 之間所有通道像素差平方的平均值：

   $$MSE = \\frac{{1}}{{3MN}} \\sum_{{c=1}}^3 \\sum_{{i=0}}^{{M-1}} \\sum_{{j=0}}^{{N-1}} [I(i,j,c)-K(i,j,c)]^2$$

2. **峰值信噪比 (PSNR)**：

   $$PSNR = 10 \\log_{{10}}\\left(\\frac{{MAX_I^2}}{{MSE}}\\right)$$

   其中 $MAX_I = 255$，代表 8 位元影像的最大像素值。MSE 越小，PSNR 通常越高。"""

    return filtered_rgb, result_text


# ==========================================
# 4. 人臉偵測模組 (MediaPipe Tasks API)
# ==========================================

def detect_faces(image_path, confidence_threshold):

    if image_path is None:
        return None, "請先上傳影像"

    img = read_image(image_path)

    if img is None:
        return None, "無法讀取影像，請確認檔案格式。"

    # BGR 轉 RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    height, width, _ = img.shape

    # MediaPipe Tasks API
    BaseOptions = mp.tasks.BaseOptions
    FaceDetector = mp.tasks.vision.FaceDetector
    FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
    RunningMode = mp.tasks.vision.RunningMode

    # 模型路徑
    model_path = "/content/blaze_face_short_range.tflite"

    # 設定人臉偵測參數
    options = FaceDetectorOptions(
        base_options=BaseOptions(
            model_asset_path=model_path
        ),
        running_mode=RunningMode.IMAGE,
        min_detection_confidence=float(confidence_threshold)
    )

    # 將 NumPy 影像轉成 MediaPipe Image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=img_rgb
    )

    # 執行人臉偵測
    with FaceDetector.create_from_options(options) as detector:

        results = detector.detect(mp_image)

    # 複製影像，準備繪製結果
    annotated_img = img_rgb.copy()

    detections = results.detections
    details = []

    for idx, detection in enumerate(detections):

        # 取得信心度
        score = float(detection.categories[0].score)

        # 取得人臉框座標
        box = detection.bounding_box

        xmin = max(0, min(width - 1, box.origin_x))
        ymin = max(0, min(height - 1, box.origin_y))

        xmax = max(0, min(width - 1,
                          box.origin_x + box.width))

        ymax = max(0, min(height - 1,
                          box.origin_y + box.height))

        if xmax <= xmin or ymax <= ymin:
            continue

        # 繪製人臉框
        cv2.rectangle(
            annotated_img,
            (xmin, ymin),
            (xmax, ymax),
            (0, 255, 0),
            2
        )

        # 顯示人臉編號及信心度
        label = f"Face {idx + 1}: {score:.2f}"

        cv2.putText(
            annotated_img,
            label,
            (xmin, max(ymin - 10, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

        # 儲存偵測資訊
        details.append(
            f"- **人臉 #{idx + 1}**：信心度 = `{score:.4f}`，"
            f"位置 = `(x:{xmin}, y:{ymin}, "
            f"w:{xmax - xmin}, h:{ymax - ymin})`"
        )

    # 統計人臉數量
    face_count = len(details)

    summary_text = (
        "### 👤 人臉偵測結果報告\n"
        f"- **偵測到總人臉數：** **{face_count}** 具\n"
        f"- **設定信心門檻：** {confidence_threshold}\n"
        + ("\n".join(details)
           if details else "- 未偵測到符合條件的人臉。")
    )

    return annotated_img, summary_text

# ==========================================
# 建構 Gradio 互動介面 (Web App)
# ==========================================
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 AI 輔助之互動式影像處理與人臉分析工具")
    gr.Markdown(
        "本系統整合了影像中繼資訊解析、色彩空間與直方圖分析、"
        "自訂卷積核濾波與 PSNR 評估，以及 MediaPipe 人臉偵測。"
    )

    with gr.Tabs():
        # Tab 1：影像載入與解析
        with gr.TabItem("1. 影像載入與解析"):
            with gr.Row():
                with gr.Column():
                    img_input_1 = gr.Image(
                        type="filepath", label="上傳影像 (PNG, JPG, BMP)"
                    )
                    btn_1 = gr.Button("解析影像", variant="primary")
                with gr.Column():
                    output_info = gr.Markdown("等待上傳...")
                    output_img_1 = gr.Image(label="原始影像預覽")
            btn_1.click(
                get_image_info,
                inputs=[img_input_1],
                outputs=[output_info, output_img_1],
            )

        # Tab 2：色彩空間與直方圖
        with gr.TabItem("2. 色彩空間與直方圖"):
            with gr.Row():
                with gr.Column():
                    img_input_2 = gr.Image(type="filepath", label="上傳影像")
                    color_mode = gr.Radio(
                        [
                            "Original (RGB)",
                            "Red Channel (R)",
                            "Green Channel (G)",
                            "Blue Channel (B)",
                            "YUV - Y (Luminance)",
                            "YUV - U (Chrominance)",
                            "YUV - V (Chrominance)",
                        ],
                        label="選擇色彩檢視模式",
                        value="Original (RGB)",
                    )
                    btn_2 = gr.Button("轉換與渲染", variant="primary")
                with gr.Column():
                    output_space_img = gr.Image(label="色彩分量影像")
                    output_hist_img = gr.Image(label="同步灰階/強度直方圖")
            btn_2.click(
                color_space_and_histogram,
                inputs=[img_input_2, color_mode],
                outputs=[output_space_img, output_hist_img],
            )

        # Tab 3：卷積濾波與 PSNR
        with gr.TabItem("3. 自訂卷積核與 PSNR 評估"):
            gr.Markdown("### 請輸入 3×3 卷積核權重 (預設為銳化濾波)")
            img_input_3 = gr.Image(
                type="filepath", label="上傳影像 (建議灰階或清晰圖)"
            )
            with gr.Row():
                k1 = gr.Number(value=0, label="K[0,0]")
                k2 = gr.Number(value=-1, label="K[0,1]")
                k3 = gr.Number(value=0, label="K[0,2]")
            with gr.Row():
                k4 = gr.Number(value=-1, label="K[1,0]")
                k5 = gr.Number(value=5, label="K[1,1]")
                k6 = gr.Number(value=-1, label="K[1,2]")
            with gr.Row():
                k7 = gr.Number(value=0, label="K[2,0]")
                k8 = gr.Number(value=-1, label="K[2,1]")
                k9 = gr.Number(value=0, label="K[2,2]")
            with gr.Row():
                normalize_chk = gr.Checkbox(
                    value=False, label="啟用權重正規化 (Normalization)"
                )
                btn_3 = gr.Button("執行卷積與計算 PSNR", variant="primary")
            with gr.Row():
                output_filtered = gr.Image(label="濾波後影像成果")
                output_psnr_report = gr.Markdown("等待運算...")

            btn_3.click(
                apply_custom_filter,
                inputs=[
                    img_input_3,
                    k1,
                    k2,
                    k3,
                    k4,
                    k5,
                    k6,
                    k7,
                    k8,
                    k9,
                    normalize_chk,
                ],
                outputs=[output_filtered, output_psnr_report],
            )

        # Tab 4：人臉偵測
        with gr.TabItem("4. 人臉偵測模組"):
            with gr.Row():
                with gr.Column():
                    img_input_4 = gr.Image(
                        type="filepath", label="上傳含人臉之影像"
                    )
                    conf_slider = gr.Slider(
                        minimum=0.1,
                        maximum=0.9,
                        value=0.5,
                        step=0.05,
                        label="推論信心度門檻 (Confidence Threshold)",
                    )
                    btn_4 = gr.Button("執行人臉偵測", variant="primary")
                with gr.Column():
                    output_face_img = gr.Image(label="人臉標記結果 (Bounding Box)")
                    output_face_report = gr.Markdown("等待偵測...")
            btn_4.click(
                detect_faces,
                inputs=[img_input_4, conf_slider],
                outputs=[output_face_img, output_face_report],
            )


# 啟動 Gradio 應用程式
# share=True 會建立公開分享連結；若不需要公開連結，可改成 share=False。
if __name__ == "__main__":
    demo.launch(share=True, debug=True, theme=gr.themes.Soft())


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://32f13e83b9a5977272.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
